# Predicting Used Car Prices with Linear Regression (from scratch)

**Goal:** predict the `SellingPrice` of a used car, and compare:
- my own **Batch Gradient Descent** linear regression (NumPy),
- my own **Stochastic Gradient Descent** (SGD) linear regression (NumPy),
- scikit-learn's `LinearRegression` and `SGDRegressor`.

**Dataset:** `vehicle_sales_clean.csv`, the cleaned used car sales data (558,723 rows, 28 columns). Each row is one car sale.

**Steps:** choose features -> group rare categories -> one-hot encode -> split into train/test -> scale -> train the models -> compare R².

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

In [23]:
df=pd.read_csv("vehicle_sales_clean.zip")

In [24]:
df.head()

,Id,Year,Make,Model,Trim,Body,Transmission,VIN,State,ConditionValue,...,PriceFlag,CarAge,PriceDiff,SaleYear,SaleQuarter,PriceDiffPct,PriceStatus,MileageBand,PriceBand,SaleMonth
0,2,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,5.0,...,normal,-1,1000.0,2014,Q4,4.878049,Above MMR,0-25K,20K-30K,2014-12
1,3,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,5.0,...,normal,-1,700.0,2014,Q4,3.365385,Above MMR,0-25K,20K-30K,2014-12
2,4,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,45.0,...,normal,1,-1900.0,2015,Q1,-5.956113,Below MMR,0-25K,30K+,2015-01
3,5,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,41.0,...,normal,0,250.0,2015,Q1,0.909091,At MMR,0-25K,20K-30K,2015-01
4,6,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,43.0,...,normal,0,1000.0,2014,Q4,1.515152,Above MMR,0-25K,30K+,2014-12


In [4]:
print(df.columns.tolist())

['Id', 'Year', 'Make', 'Model', 'Trim', 'Body', 'Transmission', 'VIN', 'State', 'ConditionValue', 'Odometer', 'Color', 'Interior', 'Seller', 'MMR', 'SellingPrice', 'SaleDate', 'ConditionScale', 'PriceFlag', 'CarAge', 'PriceDiff', 'SaleYear', 'SaleQuarter', 'PriceDiffPct', 'PriceStatus', 'MileageBand', 'PriceBand', 'SaleMonth']


## Choosing the columns (features)

The target is `SellingPrice`. We use these columns as features: `CarAge`, `Odometer`, `ConditionValue`, `Make`, `Model`, `Body`, `Transmission`, `State`, `Color`, `Interior`.

Columns that were **left out**, and why:
- **Data leakage (derived from the price):** `PriceDiff`, `PriceDiffPct`, `PriceStatus`, `PriceFlag` and `PriceBand` are calculated using `SellingPrice`. Using them would let the model "see" the answer.
- **ID-like and very high-cardinality columns:** `Id`, `VIN` and `Seller` (and `SaleDate`) have a huge number of unique values. They do not describe the car in a way the model can learn from, and would create thousands of columns.
- **Duplicated information:** `Year`, `SaleYear`, `SaleMonth` and `SaleQuarter` repeat what `CarAge` already says, and `MileageBand` repeats `Odometer`.
- **`MMR` (on purpose):** it is already a ready-made price estimate. Using it would make the model mostly copy that estimate instead of learning from the car's features.

Two extra columns are added so a linear model can follow a simple curve: `CarAge2 = (CarAge - mean)²` and `Odometer2 = (Odometer - mean)²`.

**Why centered squares?** A plain `CarAge²` is highly correlated with `CarAge` (same for `Odometer`). Correlated features make gradient descent converge very slowly. Subtracting the mean first makes the squared column measure "how far this car is from the average" instead of repeating the original column.

In [5]:
x = df[["CarAge", "Odometer", "ConditionValue",
        "Make", "Model", "Body", "Transmission",
        "State", "Color", "Interior"]]

y = df["SellingPrice"]

x = x.copy()
x["CarAge2"] = (x["CarAge"] - x["CarAge"].mean()) ** 2
x["Odometer2"] = (x["Odometer"] - x["Odometer"].mean()) ** 2

In [6]:
nums=x.select_dtypes(include="number").columns
cats=x.select_dtypes(include="object").columns

In [7]:
print(x[cats].nunique())

Make             67
Model           852
Body             46
Transmission      3
State            38
Color            20
Interior         17
dtype: int64


In [8]:
print(x.isnull().sum())

CarAge            0
Odometer          0
ConditionValue    0
Make              0
Model             0
Body              0
Transmission      0
State             0
Color             0
Interior          0
CarAge2           0
Odometer2         0
dtype: int64


## Grouping rare categories into "other"

`group_rare` keeps the most frequent values of a column and renames all the others to `"other"`.

Why it helps:
- A category with very few rows gives an **unreliable weight** (the model learns from too little data).
- Every category becomes one extra column after one-hot encoding, so fewer categories means **fewer columns** and a simpler model.

Here we keep the top 200 Models, top 50 Makes and top 30 Bodies (the `"other"` group adds one value each). The unique counts before grouping are printed above the grouping code, and after grouping below it: Make 67 → 51, Model 852 → 201, Body 46 → 31.

In [9]:
def group_rare(col,top_n):
    top=x[col].value_counts().nlargest(top_n).index
    return x[col].where(x[col].isin(top),"other")

x=x.copy()
x["Model"]=group_rare("Model",200)
x["Make"]=group_rare("Make",50)
x["Body"]=group_rare("Body",30)
print(x[cats].nunique())

Make             51
Model           201
Body             31
Transmission      3
State            38
Color            20
Interior         17
dtype: int64


## One-hot encoding

Linear regression needs numbers, so the text columns must be converted.

- We use **one-hot encoding**, not label encoding. The categories (Make, Color, State...) are **nominal**: they have no order. Label encoding (1, 2, 3...) would tell the model that, for example, one make is "bigger" than another, which is not true.
- One-hot gives each category its own 0/1 column, so the model learns a separate weight for each.
- `drop_first=True` drops one category per column. The dropped one is the reference, and this avoids redundant columns.

After encoding, the output below shows the new number of columns (359).

In [10]:
x=pd.get_dummies(x,columns=cats,drop_first=True,dtype=int)

In [11]:
print(x.shape)

(558723, 359)


In [12]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

## Scaling all the columns

`StandardScaler` is applied to **all** columns, including the one-hot (0/1) columns.

- **Why scale the numeric columns?** They have very different ranges (for example `Odometer` vs `CarAge`), and gradient descent works better when features are on a similar scale.
- **Why scale the one-hot columns too?** A rare category column is almost all zeros, so its gradient is tiny and its weight learns very slowly with gradient descent. Scaling every column makes all of them learn at a similar speed.
- **The scaler is fit on the train set only** (`fit_transform` on train, `transform` on test). The test set must act like new, unseen data. If we computed the mean and standard deviation using the test rows too, information from the test set would leak into training.

In [13]:
scaler=StandardScaler()
x_train=x_train.copy()
x_test=x_test.copy()

x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [14]:
x_train=np.array(x_train,dtype=float)
x_test=np.array(x_test,dtype=float)
y_train=np.array(y_train)
y_test=np.array(y_test)

## Batch Gradient Descent (from scratch)

The model learns a weight for each feature and one bias, by repeatedly reducing the error on the **whole training set**.

**Prediction** (n samples, X is the feature matrix):

$$\hat{y} = Xw + b$$

**Loss** (Mean Squared Error):

$$MSE = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

**Gradients** (how the loss changes with each parameter):

$$dw = \frac{1}{n} X^T (\hat{y} - y) \qquad db = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)$$

**Update** (move a small step against the gradient, `lr` = learning rate):

$$w = w - lr \cdot dw \qquad b = b - lr \cdot db$$

The code repeats these steps up to `iters` times, and prints the loss every 500 iterations so we can see it going down.

**Early stopping:** the loop stops if the loss improves by less than 1 between two iterations, because more steps would barely change the result.

In [15]:
class linearregression:
    def __init__(self,lr=0.001,iters=1000):
        self.lr=lr
        self.iters=iters
        self.weights=None
        self.bias=None

    def fit(self,x,y):
        n_sample,n_features=x.shape
        self.weights=np.zeros(n_features)
        self.bias=0
        prev_loss = None
        
        for i in range(self.iters):
            error=self.predictl(x)-y
            loss = np.mean(error ** 2)
            if prev_loss is not None and prev_loss - loss < 1:
                print("Stopped at", i, loss)
                break
            prev_loss = loss
            dw=(1/n_sample)*np.dot(x.T,error)
            db=(1/n_sample)*np.sum(error)
            self.weights -= self.lr*dw
            self.bias -= self.lr*db
            if i % 500==0:
                print(i, loss)

    def predictl(self,x):
        return np.dot(x,self.weights)+self.bias

In [16]:
m=linearregression(lr=0.3,iters=2000)
m.fit(x_train,y_train)
pred=m.predictl(x_test)
print(r2_score(y_test,pred))

0 280689901.59317464
500 20099858.01742411
1000 20096786.74578578
1500 20094620.602556545
0.7906339328973656


In [17]:
model=LinearRegression()
model.fit(x_train,y_train)
m_pred=model.predict(x_test)
print(r2_score(y_test,m_pred))

0.7906518923362325


## Stochastic Gradient Descent (from scratch)

SGD uses the **same equations** as Batch GD, but each update uses **one sample** (row) instead of the whole dataset.

For one sample $(x_i, y_i)$:

$$\hat{y}_i = x_i \cdot w + b$$

$$dw = x_i \,(\hat{y}_i - y_i) \qquad db = \hat{y}_i - y_i$$

$$w = w - lr \cdot dw \qquad b = b - lr \cdot db$$

- One **epoch** = one pass over all training rows (one update per row).
- The rows are **shuffled every epoch** (`np.random.permutation`) so the updates do not follow the same order each time.
- The updates are noisy, but there are many of them, so it needs far fewer epochs than Batch GD needs iterations.
- Pure Python loops over every row make this version slow on a large dataset.

In [18]:
class sgd:
    def __init__(self,lr=0.01,iters=50):
        self.lr=lr
        self.iters=iters
        self.weights=None
        self.bias=None

    def fit(self,x,y):
        n_sample,n_features=x.shape
        self.weights=np.zeros(n_features)
        self.bias=0
        for _ in range(self.iters):
            indcies=np.random.permutation(n_sample)
            for i in indcies:
                xi=x[i]
                yi=y[i]
                error=self.predicts(xi)-yi
                dw=xi*error
                db=error
                self.weights -= self.lr*dw
                self.bias -= self.lr*db

    def predicts(self,x):
        return np.dot(x,self.weights)+self.bias

In [20]:
my_m=sgd(lr=0.00001, iters=5)
my_m.fit(x_train,y_train)
my_pred=my_m.predicts(x_test)
print(r2_score(y_test,my_pred))

0.7896450366842661


In [21]:
s_m=SGDRegressor(tol=1e-5,max_iter=100,eta0=0.05)
s_m.fit(x_train,y_train)
s_pred=s_m.predict(x_test)
print(r2_score(y_test,s_pred))

-313495873246726.44


c:\Users\aliad\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


## Results and Conclusions

### Results (R² on the test set)

| Model | Training method | R² |
|---|---|---|
| My `linearregression` (lr=0.3, 2000 iterations) | Batch Gradient Descent | 0.7906 |
| sklearn `LinearRegression` | Closed-form solution | 0.7907 |
| My `sgd` (lr=0.001, 5 epochs) | Stochastic Gradient Descent | 0.7830 |
| sklearn `SGDRegressor` | Stochastic Gradient Descent | 0.7869 |

### Key findings

- **The model and the training method are different things.** All four are the same linear model (`y_hat = Xw + b`). Only the way the weights are found is different.
- **sklearn `LinearRegression` gives the best score (0.7907).** It solves for the optimum directly, so it is the target the other methods try to reach.
- **My Batch GD reached almost the same score (0.7906).** Its loss flattened at about 20.1 million (20,099,858 at iteration 500 and 20,094,621 at iteration 1500). The early stop did not trigger in this run, so it used all 2000 iterations.
- **The SGD versions also get close to that score** (0.7830 for mine, 0.7869 for sklearn).
- **Scaling all columns and centering the squared features made my Batch GD much closer to the optimum.** R² went from 0.6567 (only numeric columns scaled, plain squared features, lr=0.1, 5000 iterations) to 0.7906 (all columns scaled, centered squared features, lr=0.3, 2000 iterations). sklearn `LinearRegression` stayed the same (0.7907), because it uses the same information.
- **Batch GD needed about 2000 iterations, while SGD needed only 5 epochs.** SGD updates the weights after every single sample, so one epoch already contains one update per training row, many more updates than one Batch GD step.
- **Pure SGD in Python is slow on large data.** It loops over every row one by one.

### Results from earlier experiments (not shown in this notebook)

| Experiment | R² |
|---|---|
| Baseline linear regression with basic features | 0.66 |
| After adding squared features | 0.68 |
| After keeping more categories | 0.79 |
| Log-transformed target | 0.64 |
| Tree-based model (`HistGradientBoostingRegressor`) | 0.93 |

- **Log-transforming the target lowered R² here** (0.64, compared with 0.66 for the baseline).
- **Extra features helped.** Adding squared `CarAge` and `Odometer` raised R² from 0.66 to 0.68, and keeping more categories raised it to 0.79.

### Limitations

- Linear regression cannot capture **non-linear effects** (for example, price does not fall in a straight line with mileage).
- It cannot capture **interactions** (for example, the effect of age is different for a luxury car than for an economy car).

### Next steps

- A quick test with a tree-based model reached a much higher R² (0.93, from an earlier experiment not shown in this notebook). I will explore this when I study tree models.